# Feature classifier B – equal number of bona fide and spoofed recordings

This notebook trains the same simple classifier as notebook 10a (logistic regression on the
four features of notebook 09: duration, silence at the start and at the end, loudness and
bandwidth), but on a training set with as many spoofed as bona fide recordings.

**Method B:** all 2 580 bona fide recordings of ASVspoof 2019 LA train and 2 580 spoofed ones
drawn at random, 430 from each attack (A01–A06). The test sets are the same as in 10a; notebook 10c
compares both methods.

| | data |
|---|---|
| training | ASVspoof 2019 LA train, equal classes |
| test | ASVspoof 2019 LA eval; ASVspoof 2021 LA and DF (progress, eval, hidden); In-the-Wild |

**How to use it:** run notebook 09 first, then the cells from top to bottom.

**Output files**

- `results/classifier/10b_feature_classifier_equal_classes_output/` (in git):
  `training_set.csv`, `eer_table.csv`, `score_medians.csv`, `coefficients.csv`
- `outputs/10b_feature_classifier_equal_classes_output/` (not in git):
  `scores/scores_<dataset>_<subset>.csv`, `model.joblib`

**Steps**

1. Setup
2. Feature tables
3. Equal classes
4. Training
5. EER of the test sets
6. Scores per codec
7. Weights of the features
8. Saving the results

### Step 1 – Setup

**What it does:** Sets the paths and lists the training set and the test sets, with a check
that the feature table of each one exists.

**Why:** All the next cells use these paths, and a missing file is visible at once.

In [ ]:
from pathlib import Path

import joblib
import pandas as pd
from IPython.display import display

from xai_audio_deepfake_shortcuts_audit.classifier.add_scores import add_scores
from xai_audio_deepfake_shortcuts_audit.classifier.config import (
    BREAKDOWNS,
    CODEC_REFERENCES,
    FEATURE_COLUMNS,
    RANDOM_STATE,
    SUFFIX,
    TEST,
    TRAIN,
)
from xai_audio_deepfake_shortcuts_audit.classifier.eer_by_group import eer_by_group
from xai_audio_deepfake_shortcuts_audit.classifier.feature_weights import feature_weights
from xai_audio_deepfake_shortcuts_audit.classifier.score_medians import score_medians
from xai_audio_deepfake_shortcuts_audit.classifier.select_features import select_features
from xai_audio_deepfake_shortcuts_audit.classifier.train_classifier import train_classifier
from xai_audio_deepfake_shortcuts_audit.classifier.undersample_spoof import undersample_spoof
from xai_audio_deepfake_shortcuts_audit.features.read_feature_table import read_feature_table
from xai_audio_deepfake_shortcuts_audit.utils.find_repo_root import find_repo_root

REPO_ROOT = find_repo_root(Path.cwd())
FEATURES_DIR = REPO_ROOT / "outputs" / "09_feature_extraction_output" / "features"
NOTEBOOK_NAME = "10b_feature_classifier_equal_classes"
OUTPUTS_DIR = REPO_ROOT / "outputs" / f"{NOTEBOOK_NAME}_output"
SCORES_DIR = OUTPUTS_DIR / "scores"
MODEL_PATH = OUTPUTS_DIR / "model.joblib"
RESULTS_DIR = REPO_ROOT / "results" / "classifier" / f"{NOTEBOOK_NAME}_output"
SCORES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_FILES = {
    (dataset, subset): FEATURES_DIR / f"features_{dataset}_{subset}_{SUFFIX}.csv"
    for dataset, subset in [TRAIN, *TEST]
}

files = pd.DataFrame(
    [
        {
            "dataset": dataset,
            "subset": subset,
            "role": "train" if (dataset, subset) == TRAIN else "test",
            "feature_file": path.name,
            "exists": path.is_file(),
        }
        for (dataset, subset), path in FEATURE_FILES.items()
    ]
)
display(files)
print("Features:", FEATURE_COLUMNS)

### Step 2 – Feature tables

**What it does:** Reads the feature tables and counts the recordings of each class (bona fide
and spoof) in every set.

**Why:** To load the previously created feature tables and see what they consist of.

In [ ]:
tables = {key: read_feature_table(path) for key, path in FEATURE_FILES.items()}

rows = []
for (dataset, subset), table in tables.items():
    _, labels = select_features(table)
    rows.append(
        {
            "dataset": dataset,
            "subset": subset,
            "recordings": len(table),
            "usable": len(labels),
            "bonafide": int((labels == "bonafide").sum()),
            "spoof": int((labels == "spoof").sum()),
        }
    )
counts = pd.DataFrame(rows)
display(counts)

### Step 3 – Equal classes

**What it does:** Keeps all bona fide recordings of ASVspoof 2019 LA train and draws the same
number of spoofed ones at random (fixed seed), the same number from every attack. The table
shows the recordings of each class and attack before and after the draw.

**Why:** The model then learns from equal number of spoofed and bonafide recordings.
Drawing per attack keeps all six attacks equally present.

In [ ]:
usable_features, _ = select_features(tables[TRAIN])
usable = tables[TRAIN].loc[usable_features.index]
training_set = undersample_spoof(usable, RANDOM_STATE)

training_counts = pd.DataFrame(
    {
        "before": usable.groupby(["label", "attack_id"]).size(),
        "after": training_set.groupby(["label", "attack_id"]).size(),
    }
).reset_index()
display(training_counts)
display(training_set["label"].value_counts().to_frame("recordings"))

### Step 4 – Training

**What it does:** Trains the classifier on the equal training set from step 3.

**Why:** The model is the same as in notebook 10a; only the training data differ. The class
weights are now 1 for both classes, because the classes are already equal.

In [ ]:
train_features, train_labels = select_features(training_set)
model = train_classifier(train_features, train_labels)
print(f"Trained on {len(train_labels):,} recordings:", train_labels.value_counts().to_dict())

### Step 5 – EER of the test sets

**What it does:** Scores every test set with the model and computes its EER (0 % = no errors,
50 % = guessing). For some sets also per attack, codec or source.

**Why:** This is the main result: how well the four features work on recordings the model
has not seen.

In [ ]:
scored = {key: add_scores(model, tables[key]) for key in TEST}

eer_table = pd.concat(
    [
        eer_by_group(table, column).assign(dataset=dataset, subset=subset)
        for (dataset, subset), table in scored.items()
        for column in [None, *BREAKDOWNS.get((dataset, subset), [])]
    ],
    ignore_index=True,
)
eer_table = eer_table[["dataset", "subset", "group", "value", "bonafide", "spoof", "eer"]]

for group in ["all", "attack_id", "codec", "source"]:
    print("EER of every test set:" if group == "all" else f"EER per {group}:")
    display(
        eer_table[eer_table["group"] == group].style.format({"eer": "{:.2%}"}).hide(axis="index")
    )

### Step 6 – Scores per codec

**What it does:** For 2021 LA eval and for every source of 2021 DF eval, compares the
recordings after every codec with the recordings without a codec (first row): the typical
(median) score of bona fide recordings without and with the codec and their difference, the
typical score of spoofed recordings with the codec and the typical bandwidth of bona fide
recordings with the codec. All 2021 LA recordings come from ASVspoof 2019 LA eval, so their
source is shown as `asvspoof`.

**Why:** Bona fide recordings should get high scores whatever the codec. A clearly negative
`shift_bonafide` means that the model takes the codec for spoofing – a shortcut. Comparing
inside one source keeps the other differences between the sources out.

In [ ]:
score_table = pd.concat(
    [
        score_medians(part, reference).assign(
            # 2021 LA has no source column: all its recordings come from ASVspoof 2019 LA eval
            dataset=dataset,
            subset=subset,
            source=source or "asvspoof",
        )
        for (dataset, subset), reference in CODEC_REFERENCES.items()
        for source, part in scored[(dataset, subset)].groupby("source")
    ],
    ignore_index=True,
)
score_table = score_table.set_index(["dataset", "subset", "source"]).reset_index()

for (dataset, subset, source), table in score_table.groupby(
    ["dataset", "subset", "source"], sort=False
):
    print(f"{dataset} {subset}, source {source} – scores per codec:")
    display(
        table.style.format(
            {
                "median_score_bonafide": "{:.3f}",
                "median_score_bonafide_with_codec": "{:.3f}",
                "shift_bonafide": "{:+.3f}",
                "median_score_spoof_with_codec": "{:.3f}",
                "median_bandwidth_bonafide_with_codec": "{:.0f}",
            }
        ).hide(axis="index")
    )

### Step 7 – Weights of the features

**What it does:** Shows how much each feature counts in the model and in which direction
(towards spoof or bona fide), next to the typical (median) value of the feature in each class of the
equal training set.

**Why:** It shows which shortcut the model relies on most.

In [ ]:
coefficients = feature_weights(model, train_features, train_labels)
display(coefficients.style.format(precision=3).hide(axis="index"))
print(f"Intercept: {model[-1].intercept_[0]:+.3f}")

### Step 8 – Saving the results

**What it does:** Saves the list of drawn training recordings, the EER table, the scores per
codec
and the weights to `results/`, and the model and the score of every recording to `outputs/`.

**Why:** The tables go to git. The list of drawn recordings shows exactly what the model
learned from. The model and the scores are needed later, to score the modified recordings.

In [ ]:
SCORE_COLUMNS = ["dataset", "subset", "file_id", "label", "score"]

training_set[["dataset", "subset", "file_id", "label", "attack_id"]].to_csv(
    RESULTS_DIR / "training_set.csv", index=False
)
eer_table.to_csv(RESULTS_DIR / "eer_table.csv", index=False)
score_table.to_csv(RESULTS_DIR / "score_medians.csv", index=False)
coefficients.to_csv(RESULTS_DIR / "coefficients.csv", index=False)
joblib.dump(model, MODEL_PATH)
for (dataset, subset), table in scored.items():
    table[SCORE_COLUMNS].to_csv(SCORES_DIR / f"scores_{dataset}_{subset}.csv", index=False)

for path in [*sorted(RESULTS_DIR.iterdir()), MODEL_PATH, *sorted(SCORES_DIR.iterdir())]:
    print("Saved:", path.relative_to(REPO_ROOT))